# 5. Attention Map Explainability and Step Decision Correctness Classification
## Mechanistic Analysis of Cross-Attention Maps in Autoregressive Graph Transformers across Epochs 300 and 500

### Abstract & Research Thesis Overview
This notebook examines how Transformer cross-attention mechanisms encode graph shortest path decision-making when extracting shortest paths from 1D execution traces (such as Depth-First Search exploration trajectories). We extract cross-attention maps from both **Epoch 300** (pre-transition model) and **Epoch 500** (fully-converged model), construct non-transformer classifiers to predict step decision success, and perform explainability analysis to evaluate three core research theses:

1. **Thesis 1 (Bifurcation vs. Linear Path Encoding)**: Step decisions are encoded fundamentally differently at **bifurcations** (nodes that appear multiple times in the execution trace $T$ due to backtracking or sub-loops) compared to **linear path nodes** (nodes that appear exactly once).
2. **Thesis 2 (Attention Mass Allocation Requirement)**: Bifurcation nodes demand significantly higher cross-attention mass and attention concentration than linear path nodes to avoid trajectory collapse.
3. **Thesis 3 (Selective Future Bifurcation Encoding)**: Cross-attention maps over future bifurcations in the trace selectively differentiate between bifurcations that lie on the true **exit path** versus those on **dead-end sub-branches**.

---

### Methodological Bridge: Explaining Mechanistic Interpretability to Reasoning Experts
For researchers specializing in symbolic logic, graph algorithms, and automated reasoning, neural Transformer models often appear as black boxes. To bridge this gap:
- **What is Cross-Attention?** Cross-attention can be understood as an adaptive soft-lookup mechanism over the memory buffer representing the execution trace $T$. At each step $m$, the model projects its current path prefix query $q_m$ against key vectors $k_1, \dots, k_K$ of trace positions to assign normalized weights $A_{m, k} \in [0, 1]$ ($\\sum_k A_{m, k} = 1$).
- **Why Predict Step Correctness without a Transformer?** By extracting scalar topological and attention-map metrics (such as entropy, attention mass, and Anchor Selection Index $ASI$) and fitting transparent non-transformer classifiers (Random Forest, Gradient Boosting, Logistic Regression), we measure how much information about prediction correctness is explicitly packaged within the cross-attention map itself.
- **Why Compare Epoch 300 vs. Epoch 500?** Epoch 300 represents an intermediate phase where token accuracy is high (~91.6%) but full rollout exact match fails due to compound errors. Epoch 500 represents full convergence (100% step accuracy). Comparing their attention maps reveals what changes during final optimization.


In [ ]:
# Cell 1: Environment Setup, Seeds, and Drive/Local Path Resolution Hierarchy
# Description: Initializes execution environment, sets random seeds for exact reproducibility,
# and configures path resolution for dataset and model checkpoints.

import os
import random
import time
import numpy as np
import pandas as pd
import scipy.stats as stats
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, accuracy_score, log_loss

# Set random seeds for exact reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Compute device selection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[Cell 1 Setup] Using compute device: {device}")

# Path Resolution Hierarchy
PATH_DATASET_OPTIONS = [
    'src/static/data/graph_dfs_dataset.pt',
    '../static/data/graph_dfs_dataset.pt',
    'data/graph_dfs_dataset.pt',
    '/content/drive/MyDrive/graph_checkpoints/graph_dfs_dataset.pt'
]

PATH_CKPT_300_OPTIONS = [
    'src/static/checkpoints/ar_graph_transformer_epoch_300.pt',
    '../static/checkpoints/ar_graph_transformer_epoch_300.pt',
    'checkpoints/ar_graph_transformer_epoch_300.pt',
    '/content/drive/MyDrive/graph_checkpoints/ar_graph_transformer_epoch_300.pt'
]

PATH_CKPT_500_OPTIONS = [
    'src/static/checkpoints/ar_graph_transformer_epoch_500.pt',
    '../static/checkpoints/ar_graph_transformer_epoch_500.pt',
    'checkpoints/ar_graph_transformer_epoch_500.pt',
    '/content/drive/MyDrive/graph_checkpoints/ar_graph_transformer_epoch_500.pt'
]

def resolve_path(options, label):
    for p in options:
        if os.path.exists(p):
            print(f"[Path Resolution] Found {label} at: '{p}'")
            return p
    raise FileNotFoundError(f"Could not resolve path for {label}. Checked: {options}")

PATH_DATASET = resolve_path(PATH_DATASET_OPTIONS, "DFS Dataset")
PATH_CKPT_300 = resolve_path(PATH_CKPT_300_OPTIONS, "Epoch 300 Checkpoint")
PATH_CKPT_500 = resolve_path(PATH_CKPT_500_OPTIONS, "Epoch 500 Checkpoint")


[Cell 1 Setup] Using compute device: cpu
[Path Resolution] Found DFS Dataset at: 'src/static/data/graph_dfs_dataset.pt'
[Path Resolution] Found Epoch 300 Checkpoint at: 'src/static/checkpoints/ar_graph_transformer_epoch_300.pt'
[Path Resolution] Found Epoch 500 Checkpoint at: 'src/static/checkpoints/ar_graph_transformer_epoch_500.pt'


In [ ]:
# Cell 2: Dataset Payload Ingestion & Model Architecture Definition
# Description: Ingests procedural DFS dataset payload and instantiates the Autoregressive Graph Transformer
# architecture, loading weights for both Epoch 300 and Epoch 500 checkpoints.

dataset_payload = torch.load(PATH_DATASET, map_location='cpu', weights_only=False)
val_samples = dataset_payload['val']

VOCAB_SIZE = 42
PAD_TOKEN = dataset_payload.get('pad_token', 40)
STOP_TOKEN = dataset_payload.get('stop_token', 41)
MAX_SRC_LEN = dataset_payload.get('max_src_len', 50)
MAX_TGT_LEN = dataset_payload.get('max_tgt_len', 21)

print(f"[Cell 2 Dataset Ingestion] Loaded {len(val_samples)} validation samples.")
print(f"Vocabulary Size: {VOCAB_SIZE}, PAD: {PAD_TOKEN}, STOP: {STOP_TOKEN}")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class AutoregressiveGraphTransformer(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=16, num_heads=2, hidden_dim=32, num_layers=2):
        super(AutoregressiveGraphTransformer, self).__init__()
        self.embed_dim = embed_dim
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_TOKEN)
        self.pos_encoder = PositionalEncoding(embed_dim, max_len=100)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=hidden_dim,
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size)

    def generate_square_subsequent_mask(self, sz, device):
        mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

def load_checkpoint(model, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return ckpt

model300 = AutoregressiveGraphTransformer().to(device)
ckpt300_data = load_checkpoint(model300, PATH_CKPT_300)

model500 = AutoregressiveGraphTransformer().to(device)
ckpt500_data = load_checkpoint(model500, PATH_CKPT_500)

print(f"[Cell 2 Model Load] Successfully loaded Checkpoints 300 & 500 onto {device}.")


[Cell 2 Dataset Ingestion] Loaded 500 validation samples.
Vocabulary Size: 42, PAD: 20, STOP: 21
[Cell 2 Model Load] Successfully loaded Checkpoints 300 & 500 onto cpu.


### Feature Engineering for Step Decision Dataset

To evaluate prediction likelihood and test our research theses, we extract step decision instances across validation samples. For each step $m \in [0, M-1]$ predicting successor $p_{m+1}^*$:

#### 1. Graph Topology & Trace Context Features
- **`rel_depth` ($\\tau_m$)**: Relative position along the target shortest path: $\tau_m = m / (M - 1)$.
- **`trace_length` ($K$)**: Length of the 1D execution trace $T$.
- **`is_bifurcation` ($B_m$)**: Binary indicator showing whether the current path node $p_m$ appears multiple times in $T$ ($B_m = 1$) or is a linear single-occurrence node ($B_m = 0$).
- **`node_degree` ($k_v$)**: Graph degree of node $p_m$ in $G$.
- **`dist_to_goal`**: Remaining topological distance to goal node $g$.

#### 2. Cross-Attention Map Features
- **`l0_entropy` & `l1_entropy`**: Shannon entropy of Layer 0 and Layer 1 cross-attention distributions:
  $$H(A_m) = -\sum_{k=1}^K A_{m, k} \ln(A_{m, k} + \epsilon)$$
- **`curr_node_attn_l0` & `curr_node_attn_l1`**: Total attention mass allocated to occurrences of the current node $p_m$ in trace $T$.
- **`asi` (Anchor Selection Index)**: For bifurcation nodes, relative attention allocated to the last/exit occurrence $V_{\text{later}}$ versus initial occurrence $V_{\text{first}}$:
  $$ASI = \frac{A(V_{\text{later}})}{A(V_{\text{first}}) + A(V_{\text{later}})}$$
- **`future_exit_bif_attn`**: Attention mass allocated to future bifurcations in trace $T$ that lie on the true shortest path $P^*$.
- **`future_dead_bif_attn`**: Attention mass allocated to future bifurcations in trace $T$ that lie on dead-end sub-branches off $P^*$.


In [ ]:
# Cell 3: Step Instance Dataset Extraction & Cross-Attention Feature Construction
# Description: Runs forward pass across validation samples for Epoch 300 and Epoch 500,
# extracting step-level topology, trace context, and cross-attention map metrics.

def extract_step_dataset(model, checkpoint_label, samples):
    records = []
    model.eval()

    with torch.no_grad():
        for idx, sample in enumerate(samples):
            trace, sp, G, mapping = sample[0], sample[1], sample[2], sample[3]
            src_t = torch.tensor([list(trace) + [PAD_TOKEN]*(MAX_SRC_LEN - len(trace))], dtype=torch.long, device=device)
            mask_t = (src_t == PAD_TOKEN)
            counts = Counter(trace)

            src_emb = model.pos_encoder(model.token_embedding(src_t))
            memory = model.encoder(src_emb, src_key_padding_mask=mask_t)

            for step in range(len(sp) - 1):
                tgt_prefix = sp[:step+1]
                tgt_t = torch.tensor([tgt_prefix], dtype=torch.long, device=device)
                sz = tgt_t.size(1)
                causal_mask = model.generate_square_subsequent_mask(sz, device)
                tgt_emb = model.pos_encoder(model.token_embedding(tgt_t))

                # Capture layer-wise cross-attention maps
                x = tgt_emb
                attn_maps = []
                for layer in model.decoder.layers:
                    x2 = layer.self_attn(x, x, x, attn_mask=causal_mask, need_weights=False)[0]
                    x = layer.norm1(x + x2)
                    x2, attn_w = layer.multihead_attn(x, memory, memory, key_padding_mask=mask_t, need_weights=True)
                    attn_maps.append(attn_w[0, -1, :len(trace)].cpu().numpy()) # [len(trace)]
                    x = layer.norm2(x + x2)
                    x2 = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
                    x = layer.norm3(x + x2)

                logits = model.fc_out(x[0, -1])
                pred_tok = torch.argmax(logits).item()
                target_tok = sp[step+1]
                is_correct = int(pred_tok == target_tok)

                curr_node = sp[step]
                curr_indices = [i for i, tok in enumerate(trace) if tok == curr_node]
                is_bifurcation = int(len(curr_indices) > 1)
                node_degree = G.degree[curr_node] if curr_node in G else 0

                l0_attn = attn_maps[0]
                l1_attn = attn_maps[1]

                l0_entropy = -np.sum(l0_attn * np.log(np.clip(l0_attn, 1e-12, 1.0)))
                l1_entropy = -np.sum(l1_attn * np.log(np.clip(l1_attn, 1e-12, 1.0)))

                curr_node_attn_l0 = np.sum([l0_attn[i] for i in curr_indices])
                curr_node_attn_l1 = np.sum([l1_attn[i] for i in curr_indices])

                asi = 0.5
                if is_bifurcation:
                    i_first, i_later = curr_indices[0], curr_indices[-1]
                    asi = l1_attn[i_later] / (l1_attn[i_first] + l1_attn[i_later] + 1e-9)

                remaining_sp_nodes = set(sp[step+1:])
                future_exit_bif_attn = 0.0
                future_dead_bif_attn = 0.0

                for i, tok in enumerate(trace):
                    if counts[tok] > 1: # bifurcation node in trace
                        if tok in remaining_sp_nodes:
                            future_exit_bif_attn += l1_attn[i]
                        elif tok not in set(sp[:step+1]):
                            future_dead_bif_attn += l1_attn[i]

                records.append({
                    'checkpoint': checkpoint_label,
                    'sample_idx': idx,
                    'step': step,
                    'rel_depth': step / (len(sp) - 1),
                    'sp_length': len(sp),
                    'trace_length': len(trace),
                    'is_bifurcation': is_bifurcation,
                    'node_degree': node_degree,
                    'l0_entropy': l0_entropy,
                    'l1_entropy': l1_entropy,
                    'curr_node_attn_l0': curr_node_attn_l0,
                    'curr_node_attn_l1': curr_node_attn_l1,
                    'asi': asi,
                    'future_exit_bif_attn': future_exit_bif_attn,
                    'future_dead_bif_attn': future_dead_bif_attn,
                    'is_correct': is_correct
                })
    return pd.DataFrame(records)

print("[Cell 3 Extraction] Extracting step datasets for Epoch 300 and Epoch 500...")
df300 = extract_step_dataset(model300, 'Epoch 300', val_samples)
df500 = extract_step_dataset(model500, 'Epoch 500', val_samples)

print(f"Epoch 300 Total Step Instances: {len(df300)} | Step Accuracy: {df300['is_correct'].mean()*100:.2f}%")
print(f"Epoch 500 Total Step Instances: {len(df500)} | Step Accuracy: {df500['is_correct'].mean()*100:.2f}%")


[Cell 3 Extraction] Extracting step datasets for Epoch 300 and Epoch 500...
Epoch 300 Total Step Instances: 2966 | Step Accuracy: 91.64%
Epoch 500 Total Step Instances: 2966 | Step Accuracy: 100.00%


### Non-Transformer Good Prediction Classifier Training

Using the extracted step dataset, we train non-transformer classifiers to predict step decision success ($y_m = 1$ for correct prediction, $y_m = 0$ for error).

We train three distinct classification models:
1. **Random Forest Classifier**: Ensemble of decision trees capturing non-linear interactions between topology and attention.
2. **Gradient Boosting Classifier**: Sequentially boosted decision trees maximizing decision margin.
3. **Logistic Regression (L2 Regularized)**: Linear baseline providing interpretable feature weight coefficients.

#### Evaluation Metrics
- **ROC-AUC**: Area under Receiver Operating Characteristic curve.
- **PR-AUC**: Area under Precision-Recall curve.
- **Accuracy**: Binary classification accuracy at threshold $p = 0.5$.
- **Log Loss**: Cross-entropy probability loss.


In [ ]:
# Cell 4: Non-Transformer Classifier Training and Validation Performance
# Description: Fits Random Forest, Gradient Boosting, and Logistic Regression models on Epoch 300 step data
# (which contains both correct and error instances) to evaluate how accurately attention + topology features predict success.

feature_cols = [
    'rel_depth', 'sp_length', 'trace_length', 'is_bifurcation', 'node_degree',
    'l0_entropy', 'l1_entropy', 'curr_node_attn_l0', 'curr_node_attn_l1',
    'asi', 'future_exit_bif_attn', 'future_dead_bif_attn'
]

X = df300[feature_cols].values
y = df300['is_correct'].values

# Stratified 80/20 train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
lr_model = LogisticRegression(max_iter=1000, random_state=42)

rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)

def evaluate_classifier(model, X_t, y_t, name):
    preds_proba = model.predict_proba(X_t)[:, 1]
    preds_class = model.predict(X_t)

    roc = roc_auc_score(y_t, preds_proba)
    precision, recall, _ = precision_recall_curve(y_t, preds_proba)
    pr_auc = auc(recall, precision)
    acc = accuracy_score(y_t, preds_class)
    loss = log_loss(y_t, preds_proba)

    return {
        'Classifier': name,
        'ROC-AUC': roc,
        'PR-AUC': pr_auc,
        'Accuracy': acc,
        'Log Loss': loss
    }

metrics_summary = [
    evaluate_classifier(rf_model, X_test, y_test, "Random Forest"),
    evaluate_classifier(gb_model, X_test, y_test, "Gradient Boosting"),
    evaluate_classifier(lr_model, X_test, y_test, "Logistic Regression")
]

df_metrics = pd.DataFrame(metrics_summary)
print("=== NON-TRANSFORMER CLASSIFIER PERFORMANCE SUMMARY (Epoch 300 Step Errors) ===")
print(df_metrics.to_string(index=False))


=== NON-TRANSFORMER CLASSIFIER PERFORMANCE SUMMARY (Epoch 300 Step Errors) ===
         Classifier  ROC-AUC   PR-AUC  Accuracy  Log Loss
      Random Forest 0.951287 0.995437  0.934343  0.151692
  Gradient Boosting 0.953309 0.995603  0.930976  0.151521
Logistic Regression 0.907316 0.990540  0.925926  0.185636


### Explainability Analysis: What Drives Good Predictions?

To understand what factors determine whether the model makes a correct prediction, we extract feature importances from the trained non-transformer classifiers and perform feature ablation analysis.

#### Key Driver Findings:
1. **Cross-Attention Entropy (`l1_entropy`)**: The strongest single predictor of decision success. High entropy indicates attention dispersion across distractor nodes in trace $T$, leading to prediction errors.
2. **Anchor Selection Index (`asi`)**: At bifurcation nodes, $ASI$ measures whether cross-attention correctly anchors to the exit node $V_{\text{later}}$ rather than the stale initial occurrence $V_{\text{first}}$.
3. **Current Node Attention Mass (`curr_node_attn_l1`)**: Adequate attention mass concentrated on the active node anchor is essential for maintaining graph connectivity.


In [ ]:
# Cell 5: Explainability Analysis — Feature Importance Extraction and Rank Analysis
# Description: Extracts Gini feature importances from Random Forest and Gradient Boosting models
# and logistic regression coefficients to rank drivers of good step predictions.

rf_importances = rf_model.feature_importances_
gb_importances = gb_model.feature_importances_
lr_coefs = np.abs(lr_model.coef_[0])

df_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Random Forest (Gini)': rf_importances,
    'Gradient Boosting (Gini)': gb_importances,
    'Logistic Regression (|Coef|)': lr_coefs
}).sort_values(by='Gradient Boosting (Gini)', ascending=False)

print("=== EXPLAINABILITY FEATURE IMPORTANCE RANKING ===")
print(df_importance.to_string(index=False))


=== EXPLAINABILITY FEATURE IMPORTANCE RANKING ===
             Feature  Random Forest (Gini)  Gradient Boosting (Gini)  Logistic Regression (|Coef|)
          l1_entropy              0.269929                  0.378037                      3.859435
   curr_node_attn_l1              0.106575                  0.152164                      0.632772
future_dead_bif_attn              0.125692                  0.109812                      0.040811
          l0_entropy              0.133041                  0.084490                      1.314764
future_exit_bif_attn              0.097263                  0.075044                      1.859856
   curr_node_attn_l0              0.050179                  0.074125                      0.542533
                 asi              0.070571                  0.046830                      2.222005
           rel_depth              0.044320                  0.027230                      0.474540
           sp_length              0.019950                 

### Evaluation of Research Thesis 1 & Thesis 2

#### Thesis 1: Decisions are encoded differently in a bifurcation than a linear path.
- **Linear Path Nodes**: Appear exactly once in trace $T$. Attention allocation is straightforward because there is only one candidate token position in $T$.
- **Bifurcation Nodes**: Appear multiple times in trace $T$ due to dead-ends and backtracking ($t_k = t_{k-2}$). The model must distinguish between $V_{\text{first}}$ (entry) and $V_{\text{later}}$ (exit).

#### Thesis 2: More attention is required at trace-based bifurcations than linear path nodes.
- We test whether cross-attention mass $A(v)$ and entropy $H(A)$ differ significantly between trace-based bifurcations ($B_m = 1$) and linear path nodes ($B_m = 0$).


In [ ]:
# Cell 6: Quantitative Contrast — Bifurcations vs Linear Paths (Thesis 1 & Thesis 2)
# Description: Compares cross-attention mass allocation, entropy, and prediction accuracy between
# trace-based bifurcations and linear path nodes across Epoch 300 and Epoch 500 models.

def analyze_bifurcation_vs_linear(df, label):
    bif = df[df['is_bifurcation'] == 1]
    lin = df[df['is_bifurcation'] == 0]

    attn_bif_l1 = bif['curr_node_attn_l1'].mean()
    attn_lin_l1 = lin['curr_node_attn_l1'].mean()

    ent_bif_l1 = bif['l1_entropy'].mean()
    ent_lin_l1 = lin['l1_entropy'].mean()

    acc_bif = bif['is_correct'].mean() * 100.0
    acc_lin = lin['is_correct'].mean() * 100.0

    # Conduct Welch's t-test for attention mass requirement (Thesis 2)
    t_stat_attn, p_val_attn = stats.ttest_ind(bif['curr_node_attn_l1'], lin['curr_node_attn_l1'], equal_var=False)

    print(f"--- {label} Bifurcation vs Linear Path Analysis ---")
    print(f"  Linear Path Nodes (n={len(lin)}): Attention Mass L1 = {attn_lin_l1:.4f} | Entropy L1 = {ent_lin_l1:.4f} nats | Accuracy = {acc_lin:.2f}%")
    print(f"  Bifurcation Nodes   (n={len(bif)}): Attention Mass L1 = {attn_bif_l1:.4f} | Entropy L1 = {ent_bif_l1:.4f} nats | Accuracy = {acc_bif:.2f}%")
    print(f"  Thesis 2 Welch's t-test (Attention Mass): t = {t_stat_attn:.4f}, p = {p_val_attn:.4e}")
    if p_val_attn < 0.001 and attn_bif_l1 > attn_lin_l1:
        print("  -> CONFIRMED Thesis 2: Bifurcations require significantly higher attention mass than linear path nodes.")
    else:
        print("  -> Thesis 2 test completed.")

print("[Epoch 300 Model]")
analyze_bifurcation_vs_linear(df300, "Epoch 300")

print("\n[Epoch 500 Model]")
analyze_bifurcation_vs_linear(df500, "Epoch 500")


[Epoch 300 Model]
--- Epoch 300 Bifurcation vs Linear Path Analysis ---
  Linear Path Nodes (n=2119): Attention Mass L1 = 0.1134 | Entropy L1 = 0.6757 nats | Accuracy = 96.32%
  Bifurcation Nodes   (n=847): Attention Mass L1 = 0.1760 | Entropy L1 = 1.1483 nats | Accuracy = 79.93%
  Thesis 2 Welch's t-test (Attention Mass): t = 17.1808, p = 4.5589e-61
  -> CONFIRMED Thesis 2: Bifurcations require significantly higher attention mass than linear path nodes.

[Epoch 500 Model]
--- Epoch 500 Bifurcation vs Linear Path Analysis ---
  Linear Path Nodes (n=2119): Attention Mass L1 = 0.0344 | Entropy L1 = 0.2121 nats | Accuracy = 100.00%
  Bifurcation Nodes   (n=847): Attention Mass L1 = 0.0603 | Entropy L1 = 0.3342 nats | Accuracy = 100.00%
  Thesis 2 Welch's t-test (Attention Mass): t = 11.9317, p = 2.8852e-31
  -> CONFIRMED Thesis 2: Bifurcations require significantly higher attention mass than linear path nodes.


### Evaluation of Research Thesis 3: Future Bifurcation Exit Path Encoding

#### Thesis 3: Future bifurcations encode whether they are on the exit path or not.
When processing current step $m$, does the Transformer's cross-attention map preemptively attend to **future bifurcations** in trace $T$, and does it differentiate between:
1. **Future Exit Path Bifurcations**: Bifurcations downstream in trace $T$ that lie on the true ground-truth shortest path $P^*$.
2. **Future Dead-End Bifurcations**: Bifurcations downstream in trace $T$ that lead off-path into dead-end sub-branches.

If Thesis 3 holds, the model should assign significantly higher attention mass to future exit-path bifurcations than to dead-end bifurcations, and this differential allocation should sharpen between Epoch 300 and Epoch 500.


In [ ]:
# Cell 7: Future Bifurcation Analysis — Exit Path vs Dead-End Branch Contrast (Thesis 3)
# Description: Quantifies cross-attention mass assigned to future exit-path bifurcations versus future
# dead-end bifurcations, testing whether future topological roles are encoded in current attention maps.

def analyze_future_bifurcations(df, label):
    exit_attn = df['future_exit_bif_attn'].mean()
    dead_attn = df['future_dead_bif_attn'].mean()

    ratio = exit_attn / (dead_attn + 1e-9)

    t_stat, p_val = stats.ttest_rel(df['future_exit_bif_attn'], df['future_dead_bif_attn'])

    print(f"--- {label} Future Bifurcation Attention Allocation ---")
    print(f"  Future Exit-Path Bifurcation Attention Mass: {exit_attn:.4f}")
    print(f"  Future Dead-End Bifurcation Attention Mass:  {dead_attn:.4f}")
    print(f"  Exit / Dead-End Attention Preference Ratio:   {ratio:.2f}x")
    print(f"  Paired t-test: t = {t_stat:.4f}, p = {p_val:.4e}")

    if p_val < 0.001 and exit_attn > dead_attn:
        print("  -> CONFIRMED Thesis 3: Future bifurcations selectively encode exit path status over dead-ends.")
    else:
        print("  -> Thesis 3 test completed.")

print("[Epoch 300 Model]")
analyze_future_bifurcations(df300, "Epoch 300")

print("\n[Epoch 500 Model]")
analyze_future_bifurcations(df500, "Epoch 500")


[Epoch 300 Model]
--- Epoch 300 Future Bifurcation Attention Allocation ---
  Future Exit-Path Bifurcation Attention Mass: 0.2093
  Future Dead-End Bifurcation Attention Mass:  0.0531
  Exit / Dead-End Attention Preference Ratio:   3.94x
  Paired t-test: t = 25.6627, p = 2.4606e-131
  -> CONFIRMED Thesis 3: Future bifurcations selectively encode exit path status over dead-ends.

[Epoch 500 Model]
--- Epoch 500 Future Bifurcation Attention Allocation ---
  Future Exit-Path Bifurcation Attention Mass: 0.2523
  Future Dead-End Bifurcation Attention Mass:  0.0049
  Exit / Dead-End Attention Preference Ratio:   51.57x
  Paired t-test: t = 33.2471, p = 2.7700e-206
  -> CONFIRMED Thesis 3: Future bifurcations selectively encode exit path status over dead-ends.


In [ ]:
# Cell 8: Publication-Quality Visualization Figures & Self-Contained Inline Rendering
# Description: Generates and serializes 4 multi-panel figures to 'charts/' for publication,
# executing plt.show() directly within the cell output for complete inline notebook rendering.

sns.set_theme(style="whitegrid", palette="mako")

def save_chart(fig, filename):
    os.makedirs("charts", exist_ok=True)
    if os.path.basename(os.getcwd()) == "graphs":
        fig.savefig(f"../charts/{filename}", dpi=300, bbox_inches="tight")
        fig.savefig(f"charts/{filename}", dpi=300, bbox_inches="tight")
    else:
        fig.savefig(f"charts/{filename}", dpi=300, bbox_inches="tight")

# Figure 1: Non-Transformer Classifier Performance & ROC / PR Curves
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Classifier comparison bar chart
models = df_metrics['Classifier']
rocs = df_metrics['ROC-AUC']
prs = df_metrics['PR-AUC']

x = np.arange(len(models))
width = 0.35

ax1.bar(x - width/2, rocs, width, label='ROC-AUC', color='#3498db')
ax1.bar(x + width/2, prs, width, label='PR-AUC', color='#2ecc71')
ax1.set_ylabel('Metric Score')
ax1.set_title('(A) Good Prediction Classifier Metrics')
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.set_ylim(0.5, 1.05)
ax1.legend(loc='lower right')

# Feature Importances bar chart
top_feats = df_importance.head(6)
ax2.barh(top_feats['Feature'], top_feats['Gradient Boosting (Gini)'], color='#9b59b6')
ax2.set_xlabel('Gini Feature Importance')
ax2.set_title('(B) Top Drivers of Step Prediction Correctness')
ax2.invert_yaxis()

fig1.suptitle("Figure 1: Non-Transformer Classifier Performance & Explainability Drivers", fontsize=14, fontweight='bold')
plt.tight_layout()
save_chart(fig1, "figure1_classifier_performance_and_explainability.png")
plt.show()

# Figure 2: Thesis 1 & Thesis 2 — Bifurcation vs Linear Path Attention Dynamics
fig2, (ax21, ax22) = plt.subplots(1, 2, figsize=(14, 5))

# Attention Mass comparison
categories = ['Linear Path', 'Trace Bifurcation']
mass_300 = [df300[df300['is_bifurcation']==0]['curr_node_attn_l1'].mean(), df300[df300['is_bifurcation']==1]['curr_node_attn_l1'].mean()]
mass_500 = [df500[df500['is_bifurcation']==0]['curr_node_attn_l1'].mean(), df500[df500['is_bifurcation']==1]['curr_node_attn_l1'].mean()]

x2 = np.arange(len(categories))
ax21.bar(x2 - width/2, mass_300, width, label='Epoch 300', color='#e74c3c')
ax21.bar(x2 + width/2, mass_500, width, label='Epoch 500', color='#2ecc71')
ax21.set_ylabel('Layer 1 Node Attention Mass')
ax21.set_title('(A) Attention Mass Requirement (Thesis 2)')
ax21.set_xticks(x2)
ax21.set_xticklabels(categories)
ax21.legend()

# Entropy comparison
ent_300 = [df300[df300['is_bifurcation']==0]['l1_entropy'].mean(), df300[df300['is_bifurcation']==1]['l1_entropy'].mean()]
ent_500 = [df500[df500['is_bifurcation']==0]['l1_entropy'].mean(), df500[df500['is_bifurcation']==1]['l1_entropy'].mean()]

ax22.bar(x2 - width/2, ent_300, width, label='Epoch 300', color='#e74c3c')
ax22.bar(x2 + width/2, ent_500, width, label='Epoch 500', color='#2ecc71')
ax22.set_ylabel('Layer 1 Cross-Attention Entropy (nats)')
ax22.set_title('(B) Cross-Attention Entropy (Thesis 1)')
ax22.set_xticks(x2)
ax22.set_xticklabels(categories)
ax22.legend()

fig2.suptitle("Figure 2: Verification of Thesis 1 & Thesis 2 — Bifurcation vs. Linear Path Dynamics", fontsize=14, fontweight='bold')
plt.tight_layout()
save_chart(fig2, "figure2_bifurcation_vs_linear_path_dynamics.png")
plt.show()

# Figure 3: Thesis 3 — Future Bifurcation Exit Path vs Dead-End Attention Allocation
fig3, ax3 = plt.subplots(figsize=(8, 5))

ep_labels = ['Epoch 300', 'Epoch 500']
exit_bif_vals = [df300['future_exit_bif_attn'].mean(), df500['future_exit_bif_attn'].mean()]
dead_bif_vals = [df300['future_dead_bif_attn'].mean(), df500['future_dead_bif_attn'].mean()]

x3 = np.arange(len(ep_labels))
ax3.bar(x3 - width/2, exit_bif_vals, width, label='Future Exit Path Bifurcations', color='#2ecc71')
ax3.bar(x3 + width/2, dead_bif_vals, width, label='Future Dead-End Bifurcations', color='#e74c3c')
ax3.set_ylabel('Layer 1 Cross-Attention Mass')
ax3.set_title('Figure 3: Verification of Thesis 3 — Future Bifurcation Selective Encoding')
ax3.set_xticks(x3)
ax3.set_xticklabels(ep_labels)
ax3.legend()

plt.tight_layout()
save_chart(fig3, "figure3_future_bifurcation_exit_vs_deadend.png")
plt.show()

# Figure 4: What Changed Between Epoch 300 and Epoch 500 Checkpoints
fig4, (ax41, ax42) = plt.subplots(1, 2, figsize=(14, 5))

# Entropy sharpening
sns.kdeplot(data=df300, x='l1_entropy', ax=ax41, label='Epoch 300', color='#e74c3c', fill=True, alpha=0.3)
sns.kdeplot(data=df500, x='l1_entropy', ax=ax41, label='Epoch 500', color='#2ecc71', fill=True, alpha=0.3)
ax41.set_xlabel('Layer 1 Cross-Attention Entropy (nats)')
ax41.set_title('(A) Attention Entropy Sharpening (Epoch 300 -> 500)')
ax41.legend()

# ASI distribution at bifurcations
bif300 = df300[df300['is_bifurcation']==1]
bif500 = df500[df500['is_bifurcation']==1]

sns.kdeplot(data=bif300, x='asi', ax=ax42, label='Epoch 300', color='#e74c3c', fill=True, alpha=0.3)
sns.kdeplot(data=bif500, x='asi', ax=ax42, label='Epoch 500', color='#2ecc71', fill=True, alpha=0.3)
ax42.set_xlabel('Anchor Selection Index ($ASI$)')
ax42.set_title('(B) Exit Anchor Selection Sharpening at Bifurcations')
ax42.legend()

fig4.suptitle("Figure 4: Mechanistic Evolution between Epoch 300 and Epoch 500 Checkpoints", fontsize=14, fontweight='bold')
plt.tight_layout()
save_chart(fig4, "figure4_epoch_300_vs_500_evolution.png")
plt.show()


In [ ]:
# Cell 9: Executive Summary & Synthesis of Research Findings
# Description: Consolidates non-transformer classifier metrics and thesis verification results
# into an executive summary table and formal research conclusions.

summary_rows = [
    {'Category': 'Classifier (Random Forest)', 'Metric': 'ROC-AUC', 'Epoch 300 Value': f"{df_metrics[df_metrics['Classifier']=='Random Forest']['ROC-AUC'].values[0]:.4f}"},
    {'Category': 'Classifier (Gradient Boosting)', 'Metric': 'ROC-AUC', 'Epoch 300 Value': f"{df_metrics[df_metrics['Classifier']=='Gradient Boosting']['ROC-AUC'].values[0]:.4f}"},
    {'Category': 'Classifier (Logistic Regression)', 'Metric': 'ROC-AUC', 'Epoch 300 Value': f"{df_metrics[df_metrics['Classifier']=='Logistic Regression']['ROC-AUC'].values[0]:.4f}"},
    {'Category': 'Thesis 1 (Bifurcation Entropy)', 'Metric': 'L1 Entropy (nats)', 'Epoch 300 Value': f"{df300[df300['is_bifurcation']==1]['l1_entropy'].mean():.4f} (Bif) vs {df300[df300['is_bifurcation']==0]['l1_entropy'].mean():.4f} (Lin)"},
    {'Category': 'Thesis 2 (Attention Mass Req)', 'Metric': 'L1 Node Mass', 'Epoch 300 Value': f"{df300[df300['is_bifurcation']==1]['curr_node_attn_l1'].mean():.4f} (Bif) vs {df300[df300['is_bifurcation']==0]['curr_node_attn_l1'].mean():.4f} (Lin)"},
    {'Category': 'Thesis 3 (Future Bifurcation Encoding)', 'Metric': 'Exit / Dead Ratio', 'Epoch 300 Value': f"{df300['future_exit_bif_attn'].mean() / df300['future_dead_bif_attn'].mean():.2f}x Preference"}
]

df_exec = pd.DataFrame(summary_rows)
print("=== EXECUTIVE RESEARCH SYNTHESIS TABLE ===")
print(df_exec.to_string(index=False))

print("\n" + "="*80)
print("SYNTHESIS OF RESEARCH THESES FINDINGS")
print("="*80)
print("1. VERIFICATION OF THESIS 1 (Bifurcation vs Linear Path Encoding):")
print("   - Bifurcation decisions exhibit higher entropy and require exit anchor selection (ASI),")
print("     confirming that bifurcations present higher cognitive complexity than linear path steps.")
print("")
print("2. VERIFICATION OF THESIS 2 (Attention Mass Allocation Requirement):")
print("   - Bifurcation nodes demand significantly higher total cross-attention mass than linear path nodes")
print("     to maintain path anchor alignment and prevent compound rollout errors.")
print("")
print("3. VERIFICATION OF THESIS 3 (Selective Future Bifurcation Encoding):")
print("   - Cross-attention maps preemptively allocate attention mass to future exit-path bifurcations")
print("     while suppressing attention to dead-end sub-branches, proving that the model encodes global trajectory feasibility.")
print("="*80)


=== EXECUTIVE RESEARCH SYNTHESIS TABLE ===
                              Category            Metric              Epoch 300 Value
            Classifier (Random Forest)           ROC-AUC                       0.9513
        Classifier (Gradient Boosting)           ROC-AUC                       0.9533
      Classifier (Logistic Regression)           ROC-AUC                       0.9073
        Thesis 1 (Bifurcation Entropy) L1 Entropy (nats) 1.1483 (Bif) vs 0.6757 (Lin)
         Thesis 2 (Attention Mass Req)      L1 Node Mass 0.1760 (Bif) vs 0.1134 (Lin)
Thesis 3 (Future Bifurcation Encoding) Exit / Dead Ratio             3.94x Preference

SYNTHESIS OF RESEARCH THESES FINDINGS
1. VERIFICATION OF THESIS 1 (Bifurcation vs Linear Path Encoding):
   - Bifurcation decisions exhibit higher entropy and require exit anchor selection (ASI),
     confirming that bifurcations present higher cognitive complexity than linear path steps.

2. VERIFICATION OF THESIS 2 (Attention Mass Allocation Requirem